# 取得所有台股代碼
官網會每日更新，但個股的變動應該不大，隔個幾年在更新即可

In [1]:
from datetime import datetime
from io import StringIO

import pandas as pd
import requests

# 證交所 ISIN 代碼表：strMode=2 上市、strMode=4 上櫃
ISIN_URLS = {
    "上市": "https://isin.twse.com.tw/isin/C_public.jsp?strMode=2",
    "上櫃": "https://isin.twse.com.tw/isin/C_public.jsp?strMode=4",
}
# 要保留的分類區塊（排除權證、ETF、ETN、特別股、TDR 等）
STOCK_SECTIONS = {"股票", "創新板"}


def fetch_isin_stocks(url: str) -> pd.DataFrame:
    """下載 ISIN 代碼表，只保留股票類別的列。"""
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    resp.encoding = "cp950"  # 網頁為 Big5/MS950 編碼
    df = pd.read_html(StringIO(resp.text), header=0)[0]

    # 表格中穿插「股票」「ETF」等分類標題列（整列各欄內容相同），用來標記每列所屬區塊
    is_section_row = df.iloc[:, 0] == df.iloc[:, 1]
    df["板別"] = df.iloc[:, 0].where(is_section_row).ffill().str.strip()
    df = df[~is_section_row & df["板別"].isin(STOCK_SECTIONS)].copy()

    # 「有價證券代號及名稱」以全形空白分隔代號與名稱
    df[["代號", "名稱"]] = df["有價證券代號及名稱"].str.split("　", n=1, expand=True)
    df["代號"] = df["代號"].str.strip()
    df["名稱"] = df["名稱"].str.strip()
    df = df.rename(columns={"國際證券辨識號碼(ISIN Code)": "ISIN"})
    return df[["代號", "名稱", "市場別", "板別", "產業別", "上市日", "ISIN", "CFICode"]]


all_stocks = pd.concat(
    [fetch_isin_stocks(url) for url in ISIN_URLS.values()], ignore_index=True
)

file_name = f"All_Taiwan_Stock_{datetime.now():%Y%m%d}.csv"
all_stocks.to_csv(file_name, index=False, encoding="utf-8-sig")  # utf-8-sig 讓 Excel 開啟不亂碼

print(f"已儲存 {file_name}，共 {len(all_stocks)} 檔")
print(all_stocks.groupby(["市場別", "板別"]).size())
all_stocks.head()

已儲存 All_Taiwan_Stock_20261119.csv，共 1987 檔
市場別      板別 
上市       股票     1058
上市臺灣創新板  創新板      31
上櫃       股票      898
dtype: int64


,代號,名稱,市場別,板別,產業別,上市日,ISIN,CFICode
0,1101,台泥,上市,股票,水泥工業,1962/02/09,TW0001101004,ESVUFR
1,1102,亞泥,上市,股票,水泥工業,1962/06/08,TW0001102002,ESVUFR
2,1103,嘉泥,上市,股票,水泥工業,1969/11/14,TW0001103000,ESVUFR
3,1104,環泥,上市,股票,水泥工業,1971/02/01,TW0001104008,ESVUFR
4,1108,幸福,上市,股票,水泥工業,1990/06/06,TW0001108009,ESVUFR


# 使用DataScrape中的功能，取得所有股票的各項相關資料
取得：大戶交易、大戶籌碼比例、融資交易、借券交易、股價

用最新的 `All_Taiwan_Stock_YYYYMMDD.csv` 中的所有代號，呼叫 `WeekTermInvestment_DataScrape` 的爬蟲取得 5 份週資料 (法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價週K線)，存到 `DataCollection` 內各自的 ABBR 資料夾。

**可以重複執行，不會重工：**
* 每檔股票分成 4 個爬蟲 (資券餘額與借券賣出餘額是同一個爬蟲)，每個爬蟲成功就立刻存檔
* 執行前先檢查每個 ABBR 資料夾：5 份 CSV 都正確且未過舊就整檔跳過；只有部分 CSV 缺少、有誤或過舊時，只重抓那幾份
* 過舊：CSV 檔名的年月日與今天相差 `MAX_AGE_DAYS` (7) 天以上。重抓成功才會刪除舊檔，失敗時舊檔保留
* 失敗紀錄存在 `All_Taiwan_Stock_Scrape_Failures.csv`，同一份資料累計失敗 `MAX_FAILS` 次後就不再重試 (例如新上市不足 3 週、不能融資融券的股票)；失敗紀錄超過 `MAX_AGE_DAYS` 天會自動清除，下一週會再試一次。刪掉這個檔案即可全部重試。可以用 Excel 開啟查看；若用 Excel 存檔造成時間格式改變，程式也能讀取，並在下次執行時自動改回標準格式
* 連續好幾檔股票全部失敗時 (可能被 Yahoo 擋下或網路斷線)，先暫停一段時間再繼續；暫停後仍然失敗就停止執行，之後再重新執行即可
* 執行中可以隨時中斷 (Kernel → Interrupt)，已存檔的資料不受影響

In [5]:
# ===== 批次蒐集：函式定義 =====
import ast
import builtins
import json
import random
import re
import time
import types
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd()  # 本 notebook 所在的資料夾 (股票清單、失敗紀錄放這裡)
DATASCRAPE_NOTEBOOK = BASE_DIR / "WeekTermInvestment_DataScrape.ipynb"
FAILURE_LOG = BASE_DIR / "All_Taiwan_Stock_Scrape_Failures.csv"
FAILURE_LOG_COLUMNS = ["代號", "資料", "失敗次數", "最後失敗時間", "最後錯誤"]
FAILURE_TIME_FORMAT = "%Y-%m-%d %H:%M:%S"  # 寫入失敗紀錄時一律用這個格式
# 讀取時可接受的「最後失敗時間」格式：CSV 用 Excel 開啟再存檔，會被改成 2026/11/20 09:21 這類格式
FAILURE_TIME_INPUT_FORMATS = (
    FAILURE_TIME_FORMAT,
    "%Y-%m-%d %H:%M",
    "%Y/%m/%d %H:%M:%S",
    "%Y/%m/%d %H:%M",
    "%Y-%m-%d",
    "%Y/%m/%d",
)


# --- 把 DataScrape 當成 package 載入 (與 DataProcess 相同的做法) ---
def _is_definition(node, namespace):
    """只保留 import、函式/類別定義、常數設定 (等號右邊不能呼叫函式，以免執行到 input() 或爬蟲)"""
    if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        return True
    if isinstance(node, (ast.Assign, ast.AnnAssign)) and node.value is not None:
        sub_nodes = list(ast.walk(node.value))
        names = [n.id for n in sub_nodes if isinstance(n, ast.Name)]
        return not any(isinstance(n, ast.Call) for n in sub_nodes) and all(
            name in namespace or hasattr(builtins, name) for name in names
        )
    return False


def load_datascrape_tools(nb_path):
    """只執行 DataScrape 裡的 import 與函式/常數定義，不會執行「輸入代碼 → 爬資料」那一格"""
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    module = types.ModuleType(Path(nb_path).stem)
    module.__file__ = str(nb_path)
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        try:
            tree = ast.parse("".join(cell["source"]))
        except SyntaxError:
            continue
        for node in tree.body:
            if _is_definition(node, module.__dict__):
                exec(compile(ast.Module(body=[node], type_ignores=[]), str(nb_path), "exec"), module.__dict__)
    return module


ds = load_datascrape_tools(DATASCRAPE_NOTEBOOK)
DATA_DIR = BASE_DIR / ds.DATA_DIR  # 各股票的資料夾 (ABBR) 都放在這裡 (DataCollection，定義在 DataScrape)
print(f"已載入 {DATASCRAPE_NOTEBOOK.name} 的工具")

# 4 個爬蟲: (爬蟲函式, 產生的檔名標籤)；資券餘額爬蟲一次回傳 2 份資料
SCRAPERS = {
    "法人買賣超": (ds.fetch_institutional_trading_week, ["Institutional_Trading_Week_8cols"]),
    "資券餘額/借券賣出": (
        ds.fetch_margin_week,
        ["Margin_Week_11cols", "Securities_Lending_and_Borrowing_Week_8cols"],
    ),
    "大戶籌碼": (ds.fetch_major_holders_week, ["Major_Holders_Week_5cols"]),
    "股價週K線": (ds.fetch_stock_price_week, [ds.PRICE_FILE_TAG]),
}
assert sorted(t for _, tags in SCRAPERS.values() for t in tags) == sorted(ds.WEEK_FILE_COLUMNS)


# --- 檢查 ABBR 資料夾內已有的 CSV ---
def latest_stock_list(base_dir=BASE_DIR):
    """回傳最新的 All_Taiwan_Stock_YYYYMMDD.csv"""
    files = sorted(Path(base_dir).glob("All_Taiwan_Stock_[0-9]*.csv"))
    files = [f for f in files if re.fullmatch(r"All_Taiwan_Stock_\d{8}\.csv", f.name)]
    if not files:
        raise FileNotFoundError("找不到 All_Taiwan_Stock_YYYYMMDD.csv，請先執行上方「取得所有台股代碼」")
    return files[-1]


def _latest_week_file(folder, abbr, file_tag):
    """回傳 (資料夾內 ABBR_{file_tag}_YYYYMMDD.csv 中日期最新的那份, 檔案日期)，沒有就回傳 (None, None)"""
    pattern = re.compile(rf"{abbr}_{file_tag}_(\d{{8}})\.csv")
    best = (None, None)
    for f in Path(folder).glob(f"{abbr}_{file_tag}_*.csv"):
        m = pattern.fullmatch(f.name)
        if not m:
            continue
        try:
            d = datetime.strptime(m.group(1), "%Y%m%d").date()
        except ValueError:
            continue
        if best[1] is None or d > best[1]:
            best = (f, d)
    return best


def week_file_status(abbr, file_tag, today, max_age_days):
    """檢查 ABBR 資料夾內這份 CSV，回傳 "ok"、"missing" (沒有檔案或內容有誤) 或 "stale" (過舊)
    過舊：檔名的年月日與 today 相差 max_age_days 天以上"""
    path, file_date = _latest_week_file(DATA_DIR / abbr, abbr, file_tag)
    if path is None:
        return "missing"
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        return "missing"
    if df.shape[1] != len(ds.WEEK_FILE_COLUMNS[file_tag]) or df.empty:
        return "missing"
    if max_age_days is not None and (today - file_date).days >= max_age_days:
        return "stale"
    return "ok"


def scrapers_to_run(abbr, today, max_age_days):
    """回傳 {爬蟲名稱: "missing" 或 "stale"}，只列出需要執行的爬蟲
    (它產生的 CSV 只要有一份缺少/有誤就是 missing；都在但有過舊的就是 stale)"""
    need = {}
    for name, (_, tags) in SCRAPERS.items():
        statuses = {week_file_status(abbr, t, today, max_age_days) for t in tags}
        if "missing" in statuses:
            need[name] = "missing"
        elif "stale" in statuses:
            need[name] = "stale"
    return need


# --- 失敗紀錄 (每次失敗都立刻寫檔，執行中斷也不會遺失) ---
def _parse_failure_time(value):
    """把「最後失敗時間」轉成 datetime；接受 FAILURE_TIME_INPUT_FORMATS 等多種格式，無法解析就回傳 None"""
    if pd.isna(value):
        return None
    text = str(value).strip()
    for fmt in FAILURE_TIME_INPUT_FORMATS:
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass
    ts = pd.to_datetime(text, errors="coerce")  # 其他格式交給 pandas 判斷
    return None if pd.isna(ts) else ts.to_pydatetime()


def _to_int(value, default=0):
    n = pd.to_numeric(value, errors="coerce")
    return default if pd.isna(n) else int(n)


def load_failure_log():
    """讀取失敗紀錄，並把欄位統一成程式寫入的格式 (之後存檔也一律是這個格式)
    無法解析的失敗時間保留原字串，在 scrape_all_stocks 中會當成過期清除 (下次重試)"""
    if not FAILURE_LOG.exists():
        return {}
    df = pd.read_csv(FAILURE_LOG, encoding="utf-8-sig", dtype={"代號": str, "最後失敗時間": str})
    failures = {}
    for r in df.to_dict("records"):
        t = _parse_failure_time(r["最後失敗時間"])
        if t is not None:
            r["最後失敗時間"] = t.strftime(FAILURE_TIME_FORMAT)
        r["失敗次數"] = _to_int(r["失敗次數"])
        failures[(r["代號"], r["資料"])] = r
    return failures


def save_failure_log(failures):
    df = pd.DataFrame(list(failures.values()), columns=FAILURE_LOG_COLUMNS)
    df.to_csv(FAILURE_LOG, index=False, encoding="utf-8-sig")


def _short_error(e):
    msg = f"{type(e).__name__}: {e}".replace("\n", " ")
    return msg[:300]


def _fmt_duration(seconds):
    return str(timedelta(seconds=int(seconds)))


# --- 主程式 ---
def scrape_all_stocks(
    codes,
    max_age_days=7,  # CSV 檔名的年月日與今天相差幾天以上就重抓；None 代表已有的 CSV 不論日期都算完整
    max_fails=3,  # 同一份資料累計失敗幾次後就不再重試 (失敗紀錄超過 max_age_days 天會自動清除，下次再試)
    delay_s=(1, 3),  # 每個爬蟲之間隨機等待的秒數，降低被擋的機會
    pause_after=5,  # 連續幾檔股票「全部爬蟲都失敗」就暫停
    pause_s=600,  # 暫停幾秒
    headless=True,
    timeout_s=30,
):
    today = date.today()  # 以開始執行的日期為準，跨日執行時判斷標準不會變

    # 1. 失敗紀錄：超過 max_age_days 天的舊紀錄清除 (例如上週失敗的股票，這週會再試)
    failures = load_failure_log()  # 時間格式已統一 (Excel 存檔造成的格式差異在這裡修正)
    if max_age_days is not None:
        cutoff = datetime.now() - timedelta(days=max_age_days)
        kept = {
            k: r for k, r in failures.items()
            if (t := _parse_failure_time(r["最後失敗時間"])) is not None and t > cutoff
        }
        if len(kept) != len(failures):
            print(f"清除 {len(failures) - len(kept)} 筆超過 {max_age_days} 天或時間無法辨識的失敗紀錄")
            failures = kept
    if FAILURE_LOG.exists():
        save_failure_log(failures)  # 以統一的時間格式重新存檔

    # 2. 檢查已有的資料，排出待辦清單
    todo = []
    n_complete = n_gave_up = n_missing = n_stale = 0
    for abbr in codes:
        need = scrapers_to_run(abbr, today, max_age_days)
        if not need:
            n_complete += 1
            continue
        retry = [n for n in need if failures.get((abbr, n), {}).get("失敗次數", 0) < max_fails]
        n_gave_up += len(need) - len(retry)
        n_missing += sum(need[n] == "missing" for n in retry)
        n_stale += sum(need[n] == "stale" for n in retry)
        if retry:
            todo.append((abbr, retry))
    n_jobs = n_missing + n_stale
    age_note = f"，過舊 = 檔名日期與今天相差 {max_age_days} 天以上" if max_age_days is not None else ""
    print(
        f"共 {len(codes)} 檔：{n_complete} 檔資料完整 (跳過)，{len(todo)} 檔待蒐集 "
        f"({n_jobs} 個爬蟲：缺少 {n_missing}、過舊 {n_stale}{age_note})，"
        f"{n_gave_up} 個爬蟲已失敗 {max_fails} 次不再重試"
    )

    # 3. 逐檔蒐集
    t_start = time.monotonic()
    jobs_done = 0
    n_ok_stocks = 0
    all_failed_streak = 0
    paused = False
    for i, (abbr, names) in enumerate(todo, 1):
        results = {}
        for name in names:
            fetch, tags = SCRAPERS[name]
            try:
                out = fetch(abbr, headless=headless, timeout_s=timeout_s)
                dfs = out if isinstance(out, tuple) else (out,)
                ds.clear_week_csv(abbr, tags, base_dir=DATA_DIR)  # 抓取成功後才刪除這幾份的舊檔
                for tag, df in zip(tags, dfs):
                    ds.save_week_csv(df, abbr, tag, base_dir=DATA_DIR)
                if failures.pop((abbr, name), None) is not None:
                    save_failure_log(failures)
                results[name] = "✔"
            except Exception as e:
                old = failures.get((abbr, name), {})
                failures[(abbr, name)] = {
                    "代號": abbr,
                    "資料": name,
                    "失敗次數": old.get("失敗次數", 0) + 1,
                    "最後失敗時間": datetime.now().strftime(FAILURE_TIME_FORMAT),
                    "最後錯誤": _short_error(e),
                }
                save_failure_log(failures)
                results[name] = "✘"
            jobs_done += 1
            time.sleep(random.uniform(*delay_s))

        # 進度與預估剩餘時間
        elapsed = time.monotonic() - t_start
        eta = elapsed / jobs_done * (n_jobs - jobs_done)
        status = " ".join(f"{n}{r}" for n, r in results.items())
        print(f"[{i}/{len(todo)}] {abbr} {status} | 已用 {_fmt_duration(elapsed)}，預估剩餘 {_fmt_duration(eta)}")

        # 連續多檔全部失敗：可能被擋或斷線，先暫停；暫停後仍全部失敗就停止
        if "✔" in results.values():
            n_ok_stocks += "✘" not in results.values()
            all_failed_streak, paused = 0, False
        else:
            all_failed_streak += 1
            if all_failed_streak >= pause_after:
                if paused:
                    print(f"⚠ 暫停後仍連續 {all_failed_streak} 檔全部失敗，停止執行。請稍後再重新執行 (已完成的不會重抓)。")
                    break
                print(f"⚠ 連續 {all_failed_streak} 檔全部失敗，可能被網站擋下或網路斷線，暫停 {pause_s // 60} 分鐘…")
                time.sleep(pause_s)
                all_failed_streak, paused = 0, True

    print(
        f"本次結束：{n_ok_stocks} 檔補齊，用時 {_fmt_duration(time.monotonic() - t_start)}。"
        f"失敗紀錄 {len(failures)} 筆，見 {FAILURE_LOG.name}"
    )

已載入 WeekTermInvestment_DataScrape.ipynb 的工具


In [ ]:
# ===== 執行：批次蒐集所有台股的 5 份週資料 (可重複執行，完整且未過舊的資料會跳過) =====
MAX_AGE_DAYS = 7  # CSV 檔名的年月日與今天相差 7 天以上就重抓；設成 None 則已有的 CSV 不論日期都算完整
MAX_FAILS = 3     # 同一份資料累計失敗幾次後就不再重試 (刪掉 All_Taiwan_Stock_Scrape_Failures.csv 可全部重試)

stock_list_csv = latest_stock_list()
codes = pd.read_csv(stock_list_csv, dtype={"代號": str})["代號"].tolist()
print(f"使用 {stock_list_csv.name}，共 {len(codes)} 檔")

scrape_all_stocks(codes, max_age_days=MAX_AGE_DAYS, max_fails=MAX_FAILS)

使用 All_Taiwan_Stock_20261119.csv，共 1987 檔
共 1987 檔：1138 檔資料完整 (跳過)，810 檔待蒐集 (3234 個爬蟲：缺少 3234、過舊 0，過舊 = 檔名日期與今天相差 7 天以上)，42 個爬蟲已失敗 3 次不再重試
[1/810] 3067 資券餘額/借券賣出✘ | 已用 0:00:43，預估剩餘 1 day, 15:30:03
[2/810] 3115 資券餘額/借券賣出✘ | 已用 0:01:23，預估剩餘 1 day, 13:40:43
✔ 120 週的開高低收都與 K 線圖資料一致
[3/810] 3128 法人買賣超✔ 資券餘額/借券賣出✔ 大戶籌碼✔ 股價週K線✔ | 已用 0:03:43，預估剩餘 1 day, 9:21:06
✔ 120 週的開高低收都與 K 線圖資料一致
[5/810] 3141 法人買賣超✔ 資券餘額/借券賣出✔ 大戶籌碼✔ 股價週K線✔ | 已用 0:09:12，預估剩餘 1 day, 11:18:23
3147.TWO「資券餘額逐日增減」週資料最新一列是 2026-09-21，但日資料最新交易日是 2026-11-19，可能是 Yahoo 的舊快取，10 秒後重新載入 (1/2)
✔ 120 週的開高低收都與 K 線圖資料一致
[6/810] 3147 法人買賣超✔ 資券餘額/借券賣出✔ 大戶籌碼✔ 股價週K線✔ | 已用 0:11:23，預估剩餘 1 day, 9:55:10
3152.TWO「資券餘額逐日增減」週資料最新一列是 2026-11-02，但日資料最新交易日是 2026-11-19，可能是 Yahoo 的舊快取，10 秒後重新載入 (1/2)
✔ 120 週的開高低收都與 K 線圖資料一致
[7/810] 3152 法人買賣超✔ 資券餘額/借券賣出✔ 大戶籌碼✔ 股價週K線✔ | 已用 0:12:43，預估剩餘 1 day, 6:58:07
✔ 120 週的開高低收都與 K 線圖資料一致
[8/810] 3158 法人買賣超✔ 資券餘額/借券賣出✘ 大戶籌碼✔ 股價週K線✔ | 已用 0:14:09，預估剩餘 1 day, 5:06:32
3162.TWO「資券餘額逐日增減」週資料最新一列是 2026-07-13，但日資料最新交易日是 2026-11-